<a href="https://colab.research.google.com/github/Suim00/hdac-team5-knhanes/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11주차 DAG와 복합표본 회귀 (9팀)
국민건강영양조사 제9기(2022-2024) 중 2024년 자료. 아침 결식(노출)과 비만(BMI 25 이상, 결과)의 **총효과** 관련성을 무보정(모형 1)·보정(모형 2) 복합표본 로지스틱 회귀로 비교한다.

안내문의 노트북 순서를 그대로 따른다.

| 절 | 내용 |
|---|---|
| 1 | Drive 연결과 정의표의 변수만 읽기 (10주차와 같은 코드) |
| 2 | 노출·결과·공변량 만들기와 `in_study` (10주차와 같은 코드) |
| 3 | 보정변수가 모두 있는 사람 표시(완전 사례)와 인원 출력 |
| 4 | 모형 1, 모형 2(모형 3)를 설계 기반으로 적합 |
| 5 | **C-2** Table 2 형식으로 결과 정리 |
| 부록 A | **C-1** DAG와 보정집합 (역할표, 그림, 검증) |
| 부록 B | **C-3** 결과 해석 초안 |
| 부록 C | 제출 전 점검 |

`in_study`가 안내문의 `in_analysis`에 해당한다.

## 1. 내 Google Drive 연결 및 자료 읽기

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

Mounted at /content/drive


In [2]:
!pip -q install pyreadstat
import pyreadstat, numpy as np, pandas as pd

need = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe',
        'wt_tot', 'kstrata', 'psu', 'town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN']

_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
lower_map = {c.lower(): c for c in meta0.column_names}
present = {v: lower_map.get(v.lower()) for v in need}
assert all(present.values()), [v for v, a in present.items() if a is None]

df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(present.values()))
df = df.rename(columns={a: v for v, a in present.items()})
print(df.shape)

(6997, 17)


### 11주차 추가 변수 읽기 (소득·신체활동·가중치 점검용)
10주차 읽기 코드(`need` 목록)는 그대로 두고, 11주차에 새로 필요한 변수만 같은 자료에서 읽어 같은 행 순서로 붙인다.
- 신체활동 `pa_aerobic`(정의표), 소득 `incm`(DAG 교란 후보) — 변수명은 정의표와 맞는지 확인한다.
- `wt_itvex`(정의표의 종합표본가중치)는 **비교용으로만** 읽는다. 분석 가중치는 10주차와 같은 `wt_tot`을 쓰고, 3절에서 두 가중치를 비교해 볼 수 있다.

In [3]:
EXTRA = {'incm': 'incm',               # 소득 사분위(개인) 1~4   ← 정의표 변수명 확인
         'pa_aerobic': 'pa_aerobic'}   # 유산소 신체활동 0/1     ← 정의표 변수명 확인

extra_present = {v: lower_map.get(a.lower()) for v, a in EXTRA.items()}
assert all(extra_present.values()), [v for v, a in extra_present.items() if a is None]

ex, _ = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(extra_present.values()))
assert len(ex) == len(df)                       # 같은 파일이므로 행 순서가 같다
for v, a in extra_present.items():
    df[v] = ex[a].values

# 비교용 가중치 (정의표의 wt_itvex). 없으면 결측으로 둔다.
wt_alt = lower_map.get('wt_itvex')
if wt_alt:
    df['wt_itvex'] = pyreadstat.read_sas7bdat(DATA_FILE, usecols=[wt_alt])[0][wt_alt].values
else:
    df['wt_itvex'] = np.nan
    print('자료에 wt_itvex 열이 없습니다 (비교 생략)')
print(df[list(EXTRA) + ['wt_itvex']].shape)

(6997, 3)


## 2. 분석용 변수 만들기
원본 행은 지우지 않고 표시변수만 만든다.

- 노출: `L_BR_FQ` 3·4 = 결식(1), 1·2 = 아침 식사(0), 9·결측 = 결측 (이용지침서 218쪽)
- 결과: `HE_obe` 4~6 = 비만(BMI 25 이상) (이용지침서 179쪽)
- 공변량: `educ` 88·99, `N_DIET` 9는 결측 처리 (이용지침서 134·232쪽)
- 교육수준 묶음(`edu4`)은 팀이 확정한 기준으로 바꿀 수 있다.

In [4]:
df['breakfast_skip'] = df['L_BR_FQ'].map({1: 0, 2: 0, 3: 1, 4: 1})
df['obese'] = np.where(df['HE_obe'].isna(), np.nan, (df['HE_obe'] >= 4).astype(float))
df['educ_c'] = df['educ'].where(~df['educ'].isin([88, 99]))
df['edu4'] = df['educ_c'].map({1: 1, 2: 1, 3: 1, 4: 2, 5: 3, 6: 4, 7: 4, 8: 4})   # 묶음은 팀이 확정
df['N_DIET_c'] = df['N_DIET'].where(~df['N_DIET'].isin([9]))
df['clu'] = df['kstrata'].astype(str) + '_' + df['psu'].astype(str)   # 집락 식별자 (층 + 조사구)

### 11주차 추가 변수 만들기
행은 지우지 않고 값만 정리한다. 소득은 1~4 외, 신체활동은 0·1 외 값을 결측으로 처리한다. 코드 값이 정의표와 다르면 아래 두 줄과 범주 이름을 바꾼다.

In [5]:
df['incm_c'] = df['incm'].where(df['incm'].isin([1, 2, 3, 4]))          # 1=하 … 4=상 (사분위)
df['pa_c'] = df['pa_aerobic'].where(df['pa_aerobic'].isin([0, 1]))      # 1=실천, 0=비실천

### 2-2. B-1 흐름도 (분석 대상 표시변수 방식)
행을 지우지 않고 `in_study`(분석 대상), `design_ok`(설계 변수 유효) 표시변수로 단계별 인원과 제외 사유를 정리한다. (10주차 노트북 코드 그대로)

In [6]:
conds = [
    ('1. 만 19세 이상', df['age'] >= 19, '연구 대상 아님(비해당)'),
    ('2. 임신 중 아님', df['HE_prg'] != 1, '제외 기준(임신)'),
    ('3. 아침식사 빈도 응답', df['L_BR_FQ'].isin([1, 2, 3, 4]), '결측(노출 정의 불가)'),
    ('4. 신장·체중·BMI 값 있음', df[['HE_ht', 'HE_wt', 'HE_BMI']].notna().all(axis=1),
     '결측(BMI 이상값 기준은 적용하지 않음)'),
    ('5. 가중치·층·집락 유효',
     df['wt_tot'].notna() & (df['wt_tot'] > 0) & df['kstrata'].notna() & df['psu'].notna(),
     '설계상 제외'),
]

cur = pd.Series(True, index=df.index)
rows = [{'단계': '0. 전체 조사 참여자', '제외 인원': 0, '남은 인원': int(cur.sum()), '사유': '출발 대상'}]
for name, cond, why in conds:
    before = int(cur.sum())
    cur = cur & cond
    rows.append({'단계': name, '제외 인원': before - int(cur.sum()), '남은 인원': int(cur.sum()), '사유': why})

df['in_study'] = cur.astype(int)      # 분석 대상 표시변수
df['design_ok'] = conds[-1][1]        # 설계 변수 유효 표시변수
flow = pd.DataFrame(rows)
print(flow.to_string(index=False))

base = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print()
print('4단계 변수별 결측:', df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().sum().to_dict())

               단계  제외 인원  남은 인원                      사유
     0. 전체 조사 참여자      0   6997                   출발 대상
      1. 만 19세 이상    964   6033           연구 대상 아님(비해당)
       2. 임신 중 아님     16   6017               제외 기준(임신)
    3. 아침식사 빈도 응답    174   5843            결측(노출 정의 불가)
4. 신장·체중·BMI 값 있음     94   5749 결측(BMI 이상값 기준은 적용하지 않음)
   5. 가중치·층·집락 유효      0   5749                  설계상 제외

4단계 변수별 결측: {'HE_ht': 94, 'HE_wt': 23, 'HE_BMI': 94}


### 2-3. 설계변수 점검
가중치 `wt_tot`, 층 `kstrata`, 집락 `psu`가 이 분석에 맞는 설계변수인지 확인한다. 분석 대상은 자료를 잘라내지 않고 도메인(부분모집단)으로 처리한다.

In [7]:
d = df[df['design_ok']].copy()        # 설계 정보가 있는 전체 (도메인 분석용)
print('설계 자료 n:', len(d), '/ 분석 대상(도메인) n:', int(d['in_study'].sum()))
print('층 수:', d['kstrata'].nunique(), '/ 집락(PSU) 수:', d['clu'].nunique())
psu_per_strata = d.groupby('kstrata')['clu'].nunique()
print('층당 PSU 수 최소:', int(psu_per_strata.min()), '(2 이상이어야 분산 추정 가능)')
print('가중치 합(추정 모집단, 명):', round(d['wt_tot'].sum()))

설계 자료 n: 6805 / 분석 대상(도메인) n: 5749
층 수: 27 / 집락(PSU) 수: 192
층당 PSU 수 최소: 2 (2 이상이어야 분산 추정 가능)
가중치 합(추정 모집단, 명): 51522035


## 3. 보정변수가 모두 있는 사람 표시(완전 사례)와 인원 출력
**보정변수 목록은 DAG(부록 A)에서 정한 총효과 보정집합(교란변수)**이며, `ADJ_SET`에 적었다. 부록 A의 코드가 이 목록이 DAG와 같은지 확인한다.

보정변수가 하나라도 결측이면 모형 2에서 빠진다. 모형 1·2·3을 **같은 사람**으로 비교하도록 보정변수가 모두 있는 사람을 `cc`로 표시한다. 행은 지우지 않는다.

가중치는 10주차와 같은 `wt_tot`을 쓴다(`WT`). 정의표에는 종합표본가중치가 `wt_itvex`로 되어 있어 아래에서 두 가중치를 비교한다. 팀이 `wt_itvex`로 확정하면 `WT` 한 줄과 10주차 코드의 `wt_tot`을 함께 바꿔야 Table 1과 Table 2의 대상이 같다.

In [8]:
# DAG(부록 A)에서 정한 총효과 보정집합 = 교란변수 8개.  키: (표시 이름, 데이터 열, 모형에서의 형태)
ADJ_SET = {
    'age':   ('나이',          'age',       '연속'),     # 1세당
    'sex':   ('성별',          'sex',       '범주'),
    'educ':  ('교육수준',      'edu4',      '범주'),
    'incm':  ('소득',          'incm_c',    '범주'),
    'area':  ('거주지역',      'town_t',    '범주'),
    'smoke': ('현재흡연',      'sm_presnt', '범주'),
    'drink': ('월간음주',      'dr_month',  '범주'),
    'pa':    ('신체활동(유산소)', 'pa_c',    '범주'),     # 역할이 애매한 변수 (모형 3에서 제외)
}
ADJ_KEYS = list(ADJ_SET)
ADJ_CONT = [v[1] for v in ADJ_SET.values() if v[2] == '연속']     # 연속형 (기준 없음)
ADJ_CAT = [v[1] for v in ADJ_SET.values() if v[2] == '범주']      # 범주형 (가장 작은 값이 기준)
ADJ = ADJ_CONT + ADJ_CAT
assert all(c in d.columns for c in ADJ), [c for c in ADJ if c not in d.columns]

In [9]:
WT = 'wt_tot'

d['cc'] = ((d['in_study'] == 1) & d['breakfast_skip'].notna() & d['obese'].notna()
           & d[ADJ].notna().all(axis=1) & (d[WT] > 0)).astype(int)

n_study, n_cc = int(d['in_study'].sum()), int(d['cc'].sum())
print(f'Table 1 분석 대상(in_study): {n_study:,}명')
print(f'완전 사례(cc, 보정변수 모두 있음): {n_cc:,}명 → 차이 {n_study - n_cc:,}명 ({(n_study - n_cc) / n_study * 100:.1f}%)')
print()
s = d[d['in_study'] == 1]
miss = s[['breakfast_skip', 'obese'] + ADJ].isna().sum()
print('분석 대상 안의 변수별 결측 n:')
print(miss[miss > 0].to_string() if (miss > 0).any() else '  결측 없음')
print()
print('새 변수 코드 점검 (분석 대상, 결측 포함 인원 분포):')
for v in ['incm_c', 'pa_c']:
    print(s[v].value_counts(dropna=False).sort_index().to_string(), '\n')
print('완전 사례의 노출별 인원:')
print(d.loc[d['cc'] == 1, 'breakfast_skip'].value_counts().sort_index().rename({0: '아침 식사(0)', 1: '결식(1)'}).to_string())
w_ = d.loc[d['cc'] == 1]
prev = (w_['wt_tot'] * w_['obese']).sum() / w_['wt_tot'].sum() * 100
print(f"\n완전 사례의 가중 비만 유병률(점추정): {prev:.1f}%  (흔한 결과 → 오즈비 해석 주의, 유병률비는 12주차)")

# 가중치 비교 (정의표 wt_itvex와 10주차 wt_tot): 집계값만 출력
if d['wt_itvex'].notna().any():
    both = (d['wt_tot'] > 0) & (d['wt_itvex'] > 0)
    print('\n가중치 비교 (설계 자료 d 기준)')
    print(f"  wt_tot > 0: {int((d['wt_tot'] > 0).sum()):,}명 / wt_itvex > 0: {int((d['wt_itvex'] > 0).sum()):,}명 / 둘 다: {int(both.sum()):,}명")
    print(f"  분석 대상 안에서 wt_itvex가 결측이거나 0인 사람: {int(((d['in_study'] == 1) & ~(d['wt_itvex'] > 0)).sum()):,}명")
    print(f"  가중치 합(명): wt_tot {d['wt_tot'].sum():,.0f} / wt_itvex {d['wt_itvex'].sum():,.0f}")

Table 1 분석 대상(in_study): 5,749명
완전 사례(cc, 보정변수 모두 있음): 5,272명 → 차이 477명 (8.3%)

분석 대상 안의 변수별 결측 n:
edu4         148
incm_c        28
sm_presnt    141
dr_month     140
pa_c         408

새 변수 코드 점검 (분석 대상, 결측 포함 인원 분포):
incm_c
1.0    1426
2.0    1434
3.0    1434
4.0    1427
NaN      28 

pa_c
0.0    2973
1.0    2368
NaN     408 

완전 사례의 노출별 인원:
breakfast_skip
아침 식사(0)    3693
결식(1)       1579

완전 사례의 가중 비만 유병률(점추정): 37.9%  (흔한 결과 → 오즈비 해석 주의, 유병률비는 12주차)

가중치 비교 (설계 자료 d 기준)
  wt_tot > 0: 6,805명 / wt_itvex > 0: 6,805명 / 둘 다: 6,805명
  분석 대상 안에서 wt_itvex가 결측이거나 0인 사람: 0명
  가중치 합(명): wt_tot 51,522,035 / wt_itvex 50,124,465


## 4. 설계 기반 회귀: 모형 1, 모형 2(모형 3)
### 4-1. 함수 (복합표본 회귀)
가중치로 회귀계수를 추정하고, 층·집락을 반영한 Taylor 선형화(샌드위치) 분산으로 표준오차와 95% 신뢰구간을 구한다. 분석 대상 밖의 사람은 행을 지우지 않고 도메인으로 다루므로, 모형에 들어가지 않아도 집락 구조에는 그대로 남는다. 신뢰구간과 p값은 자유도 = 집락 수 − 층 수의 t분포를 쓴다.

In [10]:
from scipy import stats

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def p_eq(p):
    return 'p <0.001' if p < 0.001 else f'p = {p:.3f}'

def design_matrix(dd, cont, cat):
    """상수항 + 연속형 + 범주형 더미(가장 작은 값이 기준)."""
    X = dd[cont].astype(float).copy()
    for c in cat:
        X = X.join(pd.get_dummies(dd[c].astype(int), prefix=c, drop_first=True, dtype=float))
    X.insert(0, 'const', 1.0)
    return X

def _irls(X, y, w, kind):
    """가중 회귀계수와 (계수, 적합값, 빵 부분 A) 반환. kind: 'logit' 또는 'linear'."""
    if kind == 'linear':
        A = (X * w[:, None]).T @ X
        b = np.linalg.solve(A, (X * w[:, None]).T @ y)
        return b, X @ b, A
    b = np.zeros(X.shape[1])
    for _ in range(100):
        eta = X @ b
        mu = 1 / (1 + np.exp(-eta))
        v = np.clip(mu * (1 - mu), 1e-10, None)
        A = (X * (w * v)[:, None]).T @ X
        b_new = np.linalg.solve(A, (X * (w * v)[:, None]).T @ (eta + (y - mu) / v))
        done = np.max(np.abs(b_new - b)) < 1e-10
        b = b_new
        if done:
            break
    else:
        raise RuntimeError('로지스틱 회귀가 수렴하지 않았습니다')
    mu = 1 / (1 + np.exp(-(X @ b)))
    A = (X * (w * mu * (1 - mu))[:, None]).T @ X
    return b, mu, A

def svy_glm(d, y, cont, cat, dom, kind='logit', wcol='wt_tot'):
    """설계 기반 회귀. d 전체 행(설계 정보 있는 모든 사람)을 두고 dom(True)인 사람만 모형에 넣는다."""
    s = d[dom]
    X = design_matrix(s, cont, cat)
    yy, w = s[y].astype(float).values, s[wcol].values
    b, mu, A = _irls(X.values, yy, w, kind)

    # 점수(score)를 층·집락별로 합산. 도메인 밖 행은 0으로 두되 집락에는 남긴다.
    U = pd.DataFrame(X.values * (w * (yy - mu))[:, None], index=s.index, columns=X.columns)
    U = U.reindex(d.index, fill_value=0.0)
    U['h'], U['c'] = d['kstrata'].values, d['clu'].values
    uc = U.groupby(['h', 'c'])[list(X.columns)].sum().reset_index()
    meat = np.zeros((len(b), len(b)))
    for h, g in uc.groupby('h'):
        n = len(g)
        if n > 1:
            z = g[list(X.columns)].values
            z = z - z.mean(axis=0)
            meat += n / (n - 1) * (z.T @ z)
    Ai = np.linalg.inv(A)
    se = np.sqrt(np.diag(Ai @ meat @ Ai))

    df_ = d['clu'].nunique() - d['kstrata'].nunique()            # 설계 자유도
    t = stats.t.ppf(0.975, df_)
    p = 2 * stats.t.sf(np.abs(b / se), df_)
    out = pd.DataFrame({'b': b, 'se': se, 'lo': b - t * se, 'hi': b + t * se, 'p': p}, index=X.columns)
    if kind == 'logit':
        out['OR'], out['OR_lo'], out['OR_hi'] = np.exp(out['b']), np.exp(out['lo']), np.exp(out['hi'])
    return out, int(len(s)), int(df_)

### 4-2. 모형 1·2·3 (같은 완전 사례, 설계 반영)
- 모형 1: 노출만 (무보정)
- 모형 2: 노출 + DAG 보정집합 (총효과)
- 모형 3: 모형 2에서 애매한 변수(신체활동)만 뺀 모형. 매개 경로 가능성에 대한 예비 확인이며 정식 비교는 12주차이다.

세 모형 모두 `cc == 1`인 사람으로 적합한다. 결과는 비만(BMI 25 이상) 로지스틱, 노출은 아침 결식(1) 대 아침 식사(0)이다.

In [11]:
dom_cc = d['cc'] == 1
ADJ3_KEYS = [v for v in ADJ_KEYS if v != 'pa']                          # 모형 3: 신체활동 제외
ADJ3_CONT = [ADJ_SET[v][1] for v in ADJ3_KEYS if ADJ_SET[v][2] == '연속']
ADJ3_CAT = [ADJ_SET[v][1] for v in ADJ3_KEYS if ADJ_SET[v][2] == '범주']

res1, n1, df_design = svy_glm(d, 'obese', ['breakfast_skip'], [], dom_cc, wcol=WT)
res2, n2, _ = svy_glm(d, 'obese', ['breakfast_skip'] + ADJ_CONT, ADJ_CAT, dom_cc, wcol=WT)
res3, n3, _ = svy_glm(d, 'obese', ['breakfast_skip'] + ADJ3_CONT, ADJ3_CAT, dom_cc, wcol=WT)
assert n1 == n2 == n3 == n_cc, (n1, n2, n3, n_cc)      # 세 모형이 같은 사람인지 확인

print(f'설계 자유도(집락 수 - 층 수): {df_design}')
for name, r, n in [('모형 1 (무보정)', res1, n1), ('모형 2 (보정집합)', res2, n2), ('모형 3 (신체활동 제외)', res3, n3)]:
    e = r.loc['breakfast_skip']
    print(f"{name}: n = {n:,}, 결식 OR = {e['OR']:.2f} (95% CI {e['OR_lo']:.2f}–{e['OR_hi']:.2f}), {p_eq(e['p'])}")

설계 자유도(집락 수 - 층 수): 165
모형 1 (무보정): n = 5,272, 결식 OR = 1.11 (95% CI 0.98–1.26), p = 0.091
모형 2 (보정집합): n = 5,272, 결식 OR = 1.16 (95% CI 0.99–1.35), p = 0.058
모형 3 (신체활동 제외): n = 5,272, 결식 OR = 1.17 (95% CI 1.01–1.36), p = 0.040


## 5. Table 2 형식으로 결과 정리 (C-2)
계수는 위 모형 출력에서 그대로 옮긴다. 기준 범주는 아침 식사이다.

In [12]:
def cols(r):
    e = r.loc['breakfast_skip']
    return f"{e['OR']:.2f}", f"{e['OR_lo']:.2f}–{e['OR_hi']:.2f}", fmt_p(e['p'])

models = [('모형1', res1, n1, '없음'), ('모형2', res2, n2, f'{len(ADJ_KEYS)}개'), ('모형3', res3, n3, f'{len(ADJ3_KEYS)}개(신체활동 제외)')]
ref, skip_row, n_row, ex_row, adj_row = {'노출': '아침 식사 (기준)'}, {'노출': '아침 결식'}, {'노출': '분석 인원 n'}, {'노출': 'Table 1 대비 제외 n'}, {'노출': '보정변수'}
for m, r, n, adj in models:
    c = cols(r)
    ref.update({f'{m} OR': '1.00', f'{m} 95% CI': '', f'{m} p': ''})
    skip_row.update({f'{m} OR': c[0], f'{m} 95% CI': c[1], f'{m} p': c[2]})
    n_row.update({f'{m} OR': f'{n:,}', f'{m} 95% CI': '', f'{m} p': ''})
    ex_row.update({f'{m} OR': f'{n_study - n:,}', f'{m} 95% CI': '', f'{m} p': ''})
    adj_row.update({f'{m} OR': adj, f'{m} 95% CI': '', f'{m} p': ''})
table2 = pd.DataFrame([ref, skip_row, n_row, ex_row, adj_row])
print(table2.to_string(index=False))
print()
print(table2.to_markdown(index=False))

             노출 모형1 OR 모형1 95% CI 모형1 p 모형2 OR 모형2 95% CI 모형2 p      모형3 OR 모형3 95% CI 모형3 p
     아침 식사 (기준)   1.00                    1.00                         1.00                 
          아침 결식   1.11  0.98–1.26 0.091   1.16  0.99–1.35 0.058        1.17  1.01–1.36 0.040
        분석 인원 n  5,272                   5,272                        5,272                 
Table 1 대비 제외 n    477                     477                          477                 
           보정변수     없음                      8개                  7개(신체활동 제외)                 

| 노출                | 모형1 OR   | 모형1 95% CI   | 모형1 p   | 모형2 OR   | 모형2 95% CI   | 모형2 p   | 모형3 OR           | 모형3 95% CI   | 모형3 p   |
|:--------------------|:-----------|:---------------|:----------|:-----------|:---------------|:----------|:-------------------|:---------------|:----------|
| 아침 식사 (기준)    | 1.00       |                |           | 1.00       |                |           | 1.00               |                |      

### 각주 (문서에 함께 적기)
아래 셀이 출력하는 문장을 `docs/11_regression.md` Table 2 각주에 붙인다. 숫자는 출력에서 가져온다.

In [13]:
adj_names = ', '.join(ADJ_SET[v][0] for v in ADJ_KEYS)
footnote = (
    f"OR = 오즈비, CI = 신뢰구간. 기준 범주는 아침 식사이며, 아침 결식은 10주차 정의(L_BR_FQ 3·4)를 따랐다. "
    f"가중치 {WT}, 층 kstrata, 집락 psu(층+조사구)를 적용한 복합표본 로지스틱 회귀이며, 표준오차는 Taylor 선형화로 추정하고 "
    f"95% 신뢰구간과 p값은 자유도 {df_design}(집락 수 − 층 수)의 t분포로 구하였다. "
    f"분석 대상(Table 1의 {n_study:,}명)은 자료를 삭제하지 않고 도메인으로 지정하였고, 분석 대상 밖의 사람은 도메인 밖으로 처리하였다.\n"
    f"결과는 비만(BMI 25 이상, HE_obe 4~6)이다. 모형 1은 노출만 넣은 무보정 모형, 모형 2는 DAG로 정한 총효과 보정집합({adj_names})을 보정한 모형, "
    f"모형 3은 모형 2에서 역할이 애매한 신체활동만 뺀 모형이다. 에너지 섭취와 야식·간식은 매개 후보라 보정하지 않았고, 체중 조절 노력은 역방향 가능성이 있어 보정변수로 넣지 않았다.\n"
    f"보정변수가 모두 있는 {n_cc:,}명(완전 사례)으로 세 모형을 같은 사람에게 적합하였으며, Table 1 분석 대상보다 {n_study - n_cc:,}명 적다"
    f"(보정변수 결측, 결측 값은 대체하지 않음). 비만 유병률이 높아({prev:.1f}%) 오즈비가 위험을 실제보다 크게 보일 수 있다(유병률비는 12주차). "
    f"자료: 국민건강영양조사 제9기(2022-2024) 중 2024년."
)
print(footnote)

OR = 오즈비, CI = 신뢰구간. 기준 범주는 아침 식사이며, 아침 결식은 10주차 정의(L_BR_FQ 3·4)를 따랐다. 가중치 wt_tot, 층 kstrata, 집락 psu(층+조사구)를 적용한 복합표본 로지스틱 회귀이며, 표준오차는 Taylor 선형화로 추정하고 95% 신뢰구간과 p값은 자유도 165(집락 수 − 층 수)의 t분포로 구하였다. 분석 대상(Table 1의 5,749명)은 자료를 삭제하지 않고 도메인으로 지정하였고, 분석 대상 밖의 사람은 도메인 밖으로 처리하였다.
결과는 비만(BMI 25 이상, HE_obe 4~6)이다. 모형 1은 노출만 넣은 무보정 모형, 모형 2는 DAG로 정한 총효과 보정집합(나이, 성별, 교육수준, 소득, 거주지역, 현재흡연, 월간음주, 신체활동(유산소))을 보정한 모형, 모형 3은 모형 2에서 역할이 애매한 신체활동만 뺀 모형이다. 에너지 섭취와 야식·간식은 매개 후보라 보정하지 않았고, 체중 조절 노력은 역방향 가능성이 있어 보정변수로 넣지 않았다.
보정변수가 모두 있는 5,272명(완전 사례)으로 세 모형을 같은 사람에게 적합하였으며, Table 1 분석 대상보다 477명 적다(보정변수 결측, 결측 값은 대체하지 않음). 비만 유병률이 높아(37.9%) 오즈비가 위험을 실제보다 크게 보일 수 있다(유병률비는 12주차). 자료: 국민건강영양조사 제9기(2022-2024) 중 2024년.


## 부록 A. C-1 DAG와 보정집합
**추정 대상: 총효과** (아침 결식이 비만에 미치는 전체 효과). 정의표의 노출·결과·공변량·매개 후보를 모두 노드로 넣고, 화살표마다 근거(시간 순서)를 한 줄로 적는다. 화살표와 근거는 아래 `EDGES` 한 곳에서만 정의하고, 역할표·보정집합·dagitty 입력·Mermaid 그림은 모두 여기서 자동으로 만든다. 3절의 보정변수 목록(`ADJ_SET`)이 이 DAG에서 나온 보정집합과 같은지도 여기서 검증한다.

- 근거는 시간 순서 기준으로 적었다. **선행연구 인용은 팀이 `docs/11_regression.md`의 역할 표에 추가한다.**
- 한 시점 자료라 화살표의 방향은 자료로 확인할 수 없다. 흡연·음주·신체활동처럼 역할이 애매한 변수는 가장 그럴듯한 역할(교란)로 정하고 12주차 민감도분석에서 확인한다.
- 체중 조절 노력은 비만의 결과일 수 있고(역방향) 정의표에 없는 변수라 DAG에 넣지 않고 한계와 12주차 계획에 적는다.

In [14]:
import networkx as nx

# 노드: 키 -> (표시 이름, 데이터 열/변수명, 모형에서의 형태)  형태: 노출·결과·연속·범주·미사용
NODES = {
    'skip':  ('아침 결식',            'breakfast_skip', '노출'),
    'obese': ('비만(BMI 25 이상)',    'obese',          '결과'),
    'age':   ('나이',                 'age',            '연속'),
    'sex':   ('성별',                 'sex',            '범주'),
    'educ':  ('교육수준',             'edu4',           '범주'),
    'incm':  ('소득',                 'incm_c',         '범주'),
    'area':  ('거주지역',             'town_t',         '범주'),
    'smoke': ('현재흡연',             'sm_presnt',      '범주'),
    'drink': ('월간음주',             'dr_month',       '범주'),
    'pa':    ('신체활동(유산소)',     'pa_c',           '범주'),
    'nday':  ('야식·간식 섭취',       'N_DAY',          '미사용'),   # 매개 후보, 12주차 직접효과 민감도분석용
    'nen':   ('총에너지 섭취',        'N_EN',           '미사용'),   # 매개 후보, 12주차 직접효과 민감도분석용
}
EXPO, OUTC = 'skip', 'obese'

# 화살표: (원인, 결과, 근거 한 줄)
EDGES = [
    ('age',   'skip',  '나이는 식습관보다 먼저 정해지며 연령대마다 아침 식사 습관이 다르다'),
    ('age',   'obese', '나이는 체중 상태보다 먼저 정해지며 연령에 따라 비만 비율이 다르다'),
    ('sex',   'skip',  '성별은 태어날 때 정해지며 식사 습관이 성별로 다를 수 있다'),
    ('sex',   'obese', '성별은 태어날 때 정해지며 체지방과 비만 비율이 성별로 다르다'),
    ('educ',  'skip',  '교육은 성인기 식습관 형성보다 먼저 이루어지며 식사 시간·선택에 영향을 줄 수 있다'),
    ('educ',  'obese', '교육은 체중 상태보다 먼저 이루어지며 건강 지식과 생활 습관에 영향을 줄 수 있다'),
    ('incm',  'skip',  '소득은 식사에 쓸 시간과 비용 등 식사 여건에 영향을 줄 수 있다'),
    ('incm',  'obese', '소득은 식품 선택과 생활 환경을 통해 체중에 영향을 줄 수 있다'),
    ('area',  'skip',  '거주지(동/읍·면)는 식품 접근성과 생활 리듬에 영향을 줄 수 있다'),
    ('area',  'obese', '거주지는 식품·운동 환경을 통해 체중에 영향을 줄 수 있다'),
    ('smoke', 'skip',  '흡연은 식욕과 생활 리듬에 영향을 줄 수 있다 (한 시점 자료라 시간 순서는 불확실)'),
    ('smoke', 'obese', '흡연은 식욕과 대사를 통해 체중에 영향을 줄 수 있다 (시간 순서는 불확실)'),
    ('drink', 'skip',  '음주는 다음 날 아침 식사에 영향을 줄 수 있다 (시간 순서는 불확실)'),
    ('drink', 'obese', '음주는 섭취 열량을 통해 체중에 영향을 줄 수 있다'),
    ('pa',    'skip',  '신체활동은 식습관과 함께 형성되는 생활 습관이다 (정의표: 식습관·체중 모두에 영향을 주는 공통 원인)'),
    ('pa',    'obese', '신체활동은 에너지 소비를 통해 체중에 영향을 준다'),
    ('skip',  'nday',  '아침 결식 이후 허기로 야식·간식이 늘 수 있다 (정의표: 매개 후보)'),
    ('nday',  'obese', '야식·간식은 섭취 열량을 늘려 체중에 영향을 준다'),
    ('skip',  'nen',   '아침 결식 이후 하루 총 섭취 열량이 달라질 수 있다 (정의표: 매개 후보)'),
    ('nen',   'obese', '섭취 열량은 체중에 영향을 준다'),
    ('skip',  'obese', '추정 대상: 아침 결식이 비만에 미치는 총효과'),
]

G = nx.DiGraph([(a, b) for a, b, _ in EDGES])
assert nx.is_directed_acyclic_graph(G), 'DAG에 순환이 있습니다'
assert set(NODES) == set(G.nodes), set(NODES) ^ set(G.nodes)

# 역할: 노출과 결과의 공통 원인 = 교란, 노출의 결과이면서 결과의 원인 = 매개, 노출과 결과가 함께 원인이 되는 변수 = 충돌부
anc_x, anc_y = nx.ancestors(G, EXPO), nx.ancestors(G, OUTC)
desc_x, desc_y = nx.descendants(G, EXPO), nx.descendants(G, OUTC)
def role(v):
    if v == EXPO: return '노출'
    if v == OUTC: return '결과'
    if v in anc_x and v in anc_y: return '교란'
    if v in desc_x and v in anc_y: return '매개'
    if v in desc_x and v in desc_y: return '충돌부'
    return '기타'
ROLE = {v: role(v) for v in NODES}

# 총효과의 보정집합 = 교란변수 (매개·충돌부는 보정하지 않음)
ADJ_KEYS_DAG = [v for v in NODES if ROLE[v] == '교란']
assert not any(ROLE[v] in ('매개', '충돌부') for v in ADJ_KEYS_DAG)
# 3절에서 모형에 넣은 보정변수 목록(ADJ_SET)이 DAG에서 정한 보정집합과 같은지 확인
assert ADJ_KEYS_DAG == ADJ_KEYS, (ADJ_KEYS_DAG, ADJ_KEYS)
assert all(NODES[v][1] == ADJ_SET[v][1] and NODES[v][2] == ADJ_SET[v][2] for v in ADJ_KEYS), 'NODES와 ADJ_SET의 변수명·형태가 다릅니다'

# 역할 표
rt = []
for v, (name, col, kind) in NODES.items():
    if v in (EXPO, OUTC):
        continue
    rs = ' / '.join(f'{NODES[a][0]}→{NODES[b][0]}: {why}' for a, b, why in EDGES if v in (a, b) and not (a == EXPO and b == OUTC))
    rt.append({'변수': name, '변수명': col, '역할': ROLE[v], '보정': 'O' if v in ADJ_KEYS else 'X', '근거(시간 순서)': rs, '선행연구': '(팀이 인용 추가)'})
role_table = pd.DataFrame(rt)
print(role_table.drop(columns='근거(시간 순서)').to_string(index=False))
print()
print(role_table.to_markdown(index=False))

       변수       변수명 역할 보정       선행연구
       나이       age 교란  O (팀이 인용 추가)
       성별       sex 교란  O (팀이 인용 추가)
     교육수준      edu4 교란  O (팀이 인용 추가)
       소득    incm_c 교란  O (팀이 인용 추가)
     거주지역    town_t 교란  O (팀이 인용 추가)
     현재흡연 sm_presnt 교란  O (팀이 인용 추가)
     월간음주  dr_month 교란  O (팀이 인용 추가)
신체활동(유산소)      pa_c 교란  O (팀이 인용 추가)
 야식·간식 섭취     N_DAY 매개  X (팀이 인용 추가)
  총에너지 섭취      N_EN 매개  X (팀이 인용 추가)

| 변수             | 변수명    | 역할   | 보정   | 근거(시간 순서)                                                                                                                                                                                                       | 선행연구         |
|:-----------------|:----------|:-------|:-------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------|
| 나이             | age       | 교란   | O

In [15]:
# 보정집합 검증: 후문 기준(backdoor) — 노출에서 나가는 화살표를 지우고 노출과 결과가 보정집합으로 분리되는가
def d_sep(H, x, y, z):
    if hasattr(nx, 'is_d_separator'):
        return nx.is_d_separator(H, x, y, z)
    return nx.d_separated(H, x, y, z)

H = G.copy(); H.remove_edges_from(list(G.out_edges(EXPO)))
def backdoor_ok(keys):
    return d_sep(H, {EXPO}, {OUTC}, set(keys))

assert backdoor_ok(ADJ_KEYS), '보정집합이 모든 후문 경로를 막지 못합니다'
assert not (set(ADJ_KEYS) & desc_x), '노출의 결과(매개·충돌부)가 보정집합에 들어 있습니다'
minimal = all(not backdoor_ok([k for k in ADJ_KEYS if k != v]) for v in ADJ_KEYS)
print('총효과 보정집합(교란변수):', [NODES[v][0] for v in ADJ_KEYS])
print('후문 경로를 모두 막음: True / 최소 보정집합(하나라도 빼면 막지 못함):', minimal)
print('보정하지 않는 변수:', {NODES[v][0]: ROLE[v] for v in NODES if v not in ADJ_KEYS and v not in (EXPO, OUTC)})

# dagitty.net 입력 (붙여 넣고 Causal effect identification의 adjustment set을 확인해 문서에 함께 적는다)
dagitty = 'dag {\n' + f'  {EXPO} [exposure]\n  {OUTC} [outcome]\n' + ''.join(f'  {a} -> {b}\n' for a, b, _ in EDGES) + '}'
print('\n--- dagitty.net 입력 ---')
print(dagitty)

# Mermaid 그림 (GitHub 문서의 ```mermaid 블록에 붙여 넣는다)
def mm_label(v):
    n, col, kind = NODES[v]
    return f'{v}["{n}<br/>{col}"]'
lines = ['flowchart LR'] + [f'  {mm_label(v)}' for v in NODES]
lines += [f'  {a} --> {b}' for a, b, _ in EDGES]
lines += [f'  style {EXPO} fill:#dbeafe,stroke:#2563eb', f'  style {OUTC} fill:#fee2e2,stroke:#dc2626']
lines += [f'  style {v} stroke-dasharray: 5 5' for v in NODES if ROLE[v] == '매개']
mermaid = '```mermaid\n' + '\n'.join(lines) + '\n```'
print('\n--- Mermaid ---')
print(mermaid)

총효과 보정집합(교란변수): ['나이', '성별', '교육수준', '소득', '거주지역', '현재흡연', '월간음주', '신체활동(유산소)']
후문 경로를 모두 막음: True / 최소 보정집합(하나라도 빼면 막지 못함): True
보정하지 않는 변수: {'야식·간식 섭취': '매개', '총에너지 섭취': '매개'}

--- dagitty.net 입력 ---
dag {
  skip [exposure]
  obese [outcome]
  age -> skip
  age -> obese
  sex -> skip
  sex -> obese
  educ -> skip
  educ -> obese
  incm -> skip
  incm -> obese
  area -> skip
  area -> obese
  smoke -> skip
  smoke -> obese
  drink -> skip
  drink -> obese
  pa -> skip
  pa -> obese
  skip -> nday
  nday -> obese
  skip -> nen
  nen -> obese
  skip -> obese
}

--- Mermaid ---
```mermaid
flowchart LR
  skip["아침 결식<br/>breakfast_skip"]
  obese["비만(BMI 25 이상)<br/>obese"]
  age["나이<br/>age"]
  sex["성별<br/>sex"]
  educ["교육수준<br/>edu4"]
  incm["소득<br/>incm_c"]
  area["거주지역<br/>town_t"]
  smoke["현재흡연<br/>sm_presnt"]
  drink["월간음주<br/>dr_month"]
  pa["신체활동(유산소)<br/>pa_c"]
  nday["야식·간식 섭취<br/>N_DAY"]
  nen["총에너지 섭취<br/>N_EN"]
  age --> skip
  age --> obese
  sex --> skip
  sex --> obese
  educ

### 애매한 변수: 신체활동(`pa_aerobic`)
- **모형 2 기준 역할: 교란.** 신체활동은 식습관과 체중 모두에 영향을 주는 공통 원인으로 보고 보정한다(정의표).
- **다른 역할 가능성: 매개.** 아침 결식이 피로감이나 에너지 부족을 일으켜 신체활동을 줄이고 그것이 체중에 영향을 주는 경로(결식 → 신체활동 → 비만)를 완전히 배제할 수 없다. 한 시점 자료라 방향을 자료로 가를 수 없다.
- **확인 방법:** 모형 3은 모형 2에서 신체활동만 뺀 모형이다(같은 사람). 12주차 민감도분석에서 모형 2·3의 오즈비를 정식으로 비교한다.
- 흡연·음주도 한 시점 자료라 시간 순서가 불확실하며, 같은 방식으로 12주차에 넣고 빼는 비교로 확인할 수 있다.
- 에너지 섭취·야식/간식은 매개 후보라 총효과 모형에서 제외하고, 12주차에 직접효과 민감도분석 모형에만 선택적으로 추가한다.

## 부록 B. C-3 결과 해석
아래 셀이 출력 숫자로 해석 문장의 초안을 만든다. 문장은 팀이 다듬어 `docs/11_regression.md`에 옮기고, 한계와 12주차 계획은 팀 연구에 맞게 고친다. 해석은 '관련성'으로 쓰며 인과로 쓰지 않는다.

In [16]:
e1, e2, e3 = res1.loc['breakfast_skip'], res2.loc['breakfast_skip'], res3.loc['breakfast_skip']
ci = lambda e: f"95% CI {e['OR_lo']:.2f}–{e['OR_hi']:.2f}"

# 결과: 방향과 크기
if e2['OR_lo'] > 1:
    verdict = '아침을 거르는 사람에서 비만 오즈가 더 높은 방향의 관련성이 확인되었다.'
elif e2['OR_hi'] < 1:
    verdict = '아침을 거르는 사람에서 비만 오즈가 더 낮은 방향의 관련성이 확인되었다.'
else:
    verdict = '이 자료에서는 뚜렷한 관련성을 확인하지 못했다(신뢰구간이 1을 포함).'
result_txt = (f"보정 후(모형 2), 아침 결식군은 아침 식사군보다 비만(BMI 25 이상)일 오즈가 {e2['OR']:.2f}배"
              f"({ci(e2)}, {p_eq(e2['p'])})였다. {verdict}")

# 보정 전후: 얼마나, 어느 방향으로, 왜 (DAG 연결)
chg = (e2['OR'] / e1['OR'] - 1) * 100
if abs(chg) < 5:
    why = ('DAG에서 교란으로 본 변수들이 아침 결식과 비만 모두와 관련되더라도, '
           '이 자료에서는 서로 상쇄되었거나 교란의 크기가 작았을 수 있다.')
    direction = None
elif abs(np.log(e2['OR'])) < abs(np.log(e1['OR'])):
    direction = '가까워졌다'
    why = ('DAG에서 교란으로 정한 변수들(' + adj_names + ')이 아침 결식과 비만 모두의 공통 원인이므로, '
           '보정 전 관련성의 일부는 이 변수들에 의한 교란(과대 추정)이었을 수 있다.')
else:
    direction = '서 멀어졌다'
    why = ('DAG에서 교란으로 정한 변수들(' + adj_names + ')이 아침 결식과 비만 모두의 공통 원인이므로, '
           '보정 전에는 이 변수들이 관련성을 가리고(과소 추정) 있었을 수 있다.')
move = '오즈비가 거의 변하지 않았다' if direction is None else ('오즈비가 1에 가까워졌다' if direction == '가까워졌다' else '오즈비가 1에서 멀어졌다')
change_txt = (f"무보정(모형 1) 오즈비 {e1['OR']:.2f}({ci(e1)})에서 보정(모형 2) {e2['OR']:.2f}({ci(e2)})로 "
              f"{chg:+.1f}% 변해 {move}. {why}")
pa_txt = (f"애매한 변수인 신체활동을 뺀 모형 3의 오즈비는 {e3['OR']:.2f}({ci(e3)})로, 모형 2({e2['OR']:.2f})와 "
          f"{abs(e3['OR'] - e2['OR']):.2f} 차이이다. 신체활동이 매개라면 총효과를 일부 지울 수 있어 12주차에 정식으로 비교한다.")

print('[결과]'); print(result_txt); print()
print('[보정 전후]'); print(change_txt); print(pa_txt); print()
print('[한계]')
print('- 한 시점 자료라 아침 결식과 비만의 시간 순서를 알 수 없고, 비만이 식습관 변화(체중 조절 노력 등)의 원인일 수 있는 역방향 인과를 배제할 수 없다.')
print('- 체중 조절 노력, 유전·가족력 등 측정되지 않은 교란이 남아 있을 수 있다.')
print('- 아침 식사 빈도는 자기 기입 자료이며 결식의 기준(주 1-2회 이하, 최근 1년 거의 안 함)에 따른 분류 오차가 있을 수 있다.')
print(f'- 보정변수가 모두 있는 {n_cc:,}명만 분석해 결측으로 {n_study - n_cc:,}명이 빠졌고, 결측이 무작위가 아니면 결과가 달라질 수 있다.')
print(f'- 비만 유병률이 높아({prev:.1f}%) 오즈비가 위험을 실제보다 크게 보일 수 있다.')
print()
print('[12주차 계획]')
print('- 신체활동·흡연·음주를 넣고 빼는 비교(역할이 애매한 변수), 에너지 섭취·야식/간식을 추가한 직접효과 모형.')
print('- 체중 조절 중인 사람을 뺀 분석(역방향 인과 확인), 결측 처리 방법 바꾸기, 유병률비 산출.')

[결과]
보정 후(모형 2), 아침 결식군은 아침 식사군보다 비만(BMI 25 이상)일 오즈가 1.16배(95% CI 0.99–1.35, p = 0.058)였다. 이 자료에서는 뚜렷한 관련성을 확인하지 못했다(신뢰구간이 1을 포함).

[보정 전후]
무보정(모형 1) 오즈비 1.11(95% CI 0.98–1.26)에서 보정(모형 2) 1.16(95% CI 0.99–1.35)로 +4.5% 변해 오즈비가 거의 변하지 않았다. DAG에서 교란으로 본 변수들이 아침 결식과 비만 모두와 관련되더라도, 이 자료에서는 서로 상쇄되었거나 교란의 크기가 작았을 수 있다.
애매한 변수인 신체활동을 뺀 모형 3의 오즈비는 1.17(95% CI 1.01–1.36)로, 모형 2(1.16)와 0.01 차이이다. 신체활동이 매개라면 총효과를 일부 지울 수 있어 12주차에 정식으로 비교한다.

[한계]
- 한 시점 자료라 아침 결식과 비만의 시간 순서를 알 수 없고, 비만이 식습관 변화(체중 조절 노력 등)의 원인일 수 있는 역방향 인과를 배제할 수 없다.
- 체중 조절 노력, 유전·가족력 등 측정되지 않은 교란이 남아 있을 수 있다.
- 아침 식사 빈도는 자기 기입 자료이며 결식의 기준(주 1-2회 이하, 최근 1년 거의 안 함)에 따른 분류 오차가 있을 수 있다.
- 보정변수가 모두 있는 5,272명만 분석해 결측으로 477명이 빠졌고, 결측이 무작위가 아니면 결과가 달라질 수 있다.
- 비만 유병률이 높아(37.9%) 오즈비가 위험을 실제보다 크게 보일 수 있다.

[12주차 계획]
- 신체활동·흡연·음주를 넣고 빼는 비교(역할이 애매한 변수), 에너지 섭취·야식/간식을 추가한 직접효과 모형.
- 체중 조절 중인 사람을 뺀 분석(역방향 인과 확인), 결측 처리 방법 바꾸기, 유병률비 산출.


## 부록 C. 제출 전 점검

In [17]:
# 1) DAG에 정의표 변수(나이·성별·신체활동·야식/간식·총에너지)와 노출·결과가 모두 있다
assert {'age', 'sex', 'pa', 'nday', 'nen', EXPO, OUTC} <= set(G.nodes)
# 2) 보정집합은 교란변수뿐이고 매개·충돌부는 없다 (후문 경로를 막는지 부록 A에서 검증)
assert all(ROLE[v] == '교란' for v in ADJ_KEYS) and backdoor_ok(ADJ_KEYS)
# 3) 세 모형이 같은 사람이며 완전 사례 수와 같다
assert n1 == n2 == n3 == int(d['cc'].sum())
# 4) 완전 사례는 분석 대상 안에 있다 (도메인 밖 사람은 들어오지 않음)
assert (d.loc[d['cc'] == 1, 'in_study'] == 1).all()
# 5) 설계 변수가 유효하다
assert d['wt_tot'].gt(0).all() and d['kstrata'].notna().all() and d['psu'].notna().all()
# 6) 보정변수 열이 모두 자료에 있다
assert all(c in d.columns for c in ADJ)
# 7) 개인별 행을 출력하지 않았다 (요약 표와 집계값만 출력)
print('점검 통과')

점검 통과
